## 🔰PyTorchでニューラルネットワーク基礎 #31【事前学習】

### 内容
* Qiitaの記事と連動しています
* BERTタイプ(Transformer Encoderタイプ)を利用した事前学習を行ってみる
* トークナイザーはunigram lm を利用して作成
* 学習に利用するデータ：wikipediaの日本の歴史に相当する部分をいくつか抽出
* データはJSONL形式で、トークナイズされたtextとidsをキーに持っている

**利用するファイル**
* トークナイザー：unigram_tokenizer_2k.json
* 学習用データ：train_data_unigram_2k.jsonl
* 保存モデル：unigram_2k.model

**学習データの扱い**
* 等長の系列長ベクトル
* TensorDatasetを利用する
* DataLoaderにカスタムcollator関数を使う (mlm用のcollate_fnについてはsample_29.ipynbを参照)

**重み共有によるパラメータの変化**
* 最終層の全結合層の重みとトークンの埋め込み層の重みを共有させる手法
* MLMヘッドFCのみのタイプ：1.6M程度
* MLMヘッドをBERT風タイプ：1M程度

**BERTタイプの事前学習について**
* 読み込んだ文章の一部をマスクに置き換えて、マスクの単語を予測する形でパラメータを学習していく。
* ランダムにトークンの一部を\<mask\>に変更。
* 学習するたびに\<mask\>の位置が変わるようにする。ここがポイント
* 今回はMLM (Masked Language Modeling) のみで事前学習を行う。
* 文の接続性判定であるNSP (Next Sentence Prediction)は行っていません。

**ネットワーク構造の基本形**
* モデルの構造についてはsample_30.ipynbを参照
* トークンと位置の埋め込み → Transformer Encoder → Linear (mlm_head) → 単語の予測

In [1]:
import torch
import torch.nn as nn
import random
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset  # ミニバッチの利用 
from tokenizers import Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# 事前学習のモデル名
pretrain_filename = "./model/unigram_2k.model"
# 保存したトークナイザー
tokenizer_filename = "tokenizer/unigram_tokenizer_2k.json"
tokenizer = Tokenizer.from_file(tokenizer_filename)

print("特殊トークンID:")
print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<bos>: {tokenizer.token_to_id('<bos>')}")
print(f"<eos>: {tokenizer.token_to_id('<eos>')}")
print(f"<unk>: {tokenizer.token_to_id('<unk>')}")
print(f"<mask>: {tokenizer.token_to_id('<mask>')}")
print(f"size: {tokenizer.get_vocab_size()}")

特殊トークンID:
<pad>: 0
<bos>: 1
<eos>: 2
<unk>: 3
<mask>: 4
size: 2000


### ネットワーク構造

1. モデルの設定をクラスファイルで統合した。
    ~~~python
    config = ModelConfig(tokenizer)
    model = DNN(config)
    ~~~

2. MLMHeadクラス
    * mlm_head部分のBERT風タイプ。論文よりの実装。weight tyingも追加している。
    * nn.Linear(d_model, vocab_size)という簡易版でもOK。

In [3]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.d_model = 64
        self.seq_len = 64
        self.nhead = 4
        self.dim_feedforward = 256
        self.num_layers = 6
        self.dropout = 0.1
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.mask_token_id = tokenizer.token_to_id("<mask>")
        self.bos_token_id = tokenizer.token_to_id("<bos>")
        self.eos_token_id = tokenizer.token_to_id("<eos>")
        self.unk_token_id = tokenizer.token_to_id("<unk>")

        # 特殊トークンのセット
        self.special_tokens_set = {
            self.pad_token_id,
            self.mask_token_id,
            self.bos_token_id,
            self.eos_token_id,
            self.unk_token_id,
        }
        
        # 通常トークンのリスト special_tokenを除くトークンのリスト（MLMランダム置換用）
        self.normal_tokens_list = [
            i for i in range(self.vocab_size) 
            if i not in self.special_tokens_set
        ]
        
        # 学習設定 (今回は利用しないけど使うと便利かも)
        self.batch_size = 32
        self.learning_rate = 0.001
        self.num_epochs = 100
        self.mask_prob = 0.15
        self.max_grad_norm = 1.0

In [4]:
# MLM用の出力部分
class MLMHead(nn.Module):
    def __init__(self, config: ModelConfig, embedding_weight):
        super().__init__()
        self.fc = nn.Linear(config.d_model, config.d_model)
        self.act = nn.GELU()
        self.ln = nn.LayerNorm(config.d_model)

        self.classification_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.classification_head.weight = embedding_weight  # weight tying

        self.bias = nn.Parameter(torch.zeros(config.vocab_size))

    def forward(self, x):
        x = self.fc(x)
        x = self.act(x)
        x = self.ln(x)
        x = self.classification_head(x) + self.bias
        return x

# MLM用のモデル（出力層を変更）
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        # 埋め込み層
        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size, 
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id
        )
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers, enable_nested_tensor=False)
        
        # MLM用の出力層：BERT風レイヤー　各トークン位置で語彙全体を予測
        self.mlm_head = MLMHead(config, self.token_embedding.weight)
        #self.mlm_head = nn.Linear(config.d_model, config.vocab_size)
    
    def forward(self, x):
        # マスクの作成
        src_key_padding_mask = (x == self.config.pad_token_id)
        
        # 埋め込み
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(torch.arange(x.size(1), device=x.device))
        x = tok_emb + pos_emb.unsqueeze(0)
        
        x = self.layer_norm(x)
        x = self.dropout(x)
        
        # Transformer Encoder
        h = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)
        
        # MLM予測：各位置で語彙全体での確率を出力
        logits = self.mlm_head(h)  # [batch, seq_len, vocab_size]
        
        return logits

In [5]:
config = ModelConfig(tokenizer)
model = DNN(config).to(device)

In [6]:
# 保存したモデルを復元する時
# <mask>予測させる時に使う
# checkpoint = torch.load(pretrain_filename)
# config = ModelConfig(tokenizer)
# config.__dict__.update(checkpoint["config"])
# model = DNN(config).to(device)
# model.load_state_dict(checkpoint["model_state_dict"])

In [7]:
# import torch
# import random

def mlm_sample(batch, mask_prob:float = 0.15):
    """
    batch: [(x1,), (x2,),...,(xi,),...] というタプルのリストの予定
    x1: tensor
    """

    special_tokens_set = config.special_tokens_set # {0,1,2,3,4}  # <pad>, <bos>, <eos>, <unk>, <mask>になる予定
    normal_tokens_list = config.normal_tokens_list  # range(5,10_000) # 語彙IDのspecial_tokensを除いたもの

    # 句読点も除外 (マスクに句読点を入れる結果になりやすかったので今回あえて削除しました)
    no_mask_tokens = ["、", "。", ",", ".", "，", "．"]
    no_mask_ids = set(
        token_id for token_id in [tokenizer.token_to_id(tok) for tok in no_mask_tokens]
        if token_id is not None
    )
    special_tokens_set.update(no_mask_ids)
    
    all_input_ids = []
    all_labels = []

    # 1. 各サンプルに対してループ処理
    for item in batch:
        original_ids = item[0]                        # item[0]でxiを取り出す
        input_ids = original_ids.clone()              # originalも使うのでcloneしておく
        labels = torch.full_like(original_ids, -100)  # ラベル：一旦　ID = -100 にしてから、該当部分のみ書き換える
       
        for i in range(len(original_ids)):

            # 特殊トークンの場合は飛ばす torch.Tensorをintに戻してから判定
            token_id = int(original_ids[i].item())
            if token_id in special_tokens_set:
                continue

            # 特殊トークン以外15%の確率が置き換え対象
            if random.random() < mask_prob:
                labels[i] = original_ids[i]   # labelsに元のIDを入れる (教師データになる)
                # 置き換え対象の処理 80/10/10 で マスク/他トークン/そのまま
                rand = random.random()
                if rand < 0.8:
                    input_ids[i] = config.mask_token_id # mask_id 今回はそのまま入力しているけど
                elif rand < 0.9:
                    input_ids[i] = random.choice(normal_tokens_list) # 特殊トークンID以外のトークンで置き換え
                else:
                    pass # 10%はそのまま
        
        all_input_ids.append(input_ids)
        all_labels.append(labels)

    # 2. リストをスタックして (bs, seq_Len) のテンソルに戻す
    return torch.stack(all_input_ids), torch.stack(all_labels)

In [ ]:
# データファイルの読み込みJSONLファイルを読み込む
# import pandas as pd

filename = f"./data/train_data_unigram_2k.jsonl"
df = pd.read_json(filename, lines=True)
x = torch.LongTensor(df["ids"])

In [9]:
train_data = TensorDataset(x)

# DataLoaderに渡す
train_loader = DataLoader(
    train_data, 
    batch_size=32, 
    shuffle=True, 
    collate_fn=mlm_sample,
    drop_last=True,
    num_workers=4
)

In [10]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)

In [ ]:
def accuracy(y, t):
    """
    マスク位置での予測精度を計算
    ※この関数内では勾配計算を行わないマスク位置での予測精度を計算
    """
    with torch.no_grad():
        preds = torch.argmax(y, dim=-1)
        mask = (t != -100)
        correct = (preds == t) & mask
        
        num_correct = correct.sum().item()
        num_total = mask.sum().item()
        # 系列長が64と短いので学習候補が0個になる可能性がある。予防的につけておきました。
        accuracy = (num_correct / num_total) if num_total > 0 else 0.0
        
        return accuracy

In [ ]:
LOOP = 100
model.train()
for epoch in range(LOOP):
    total_loss = 0
    total_acc = 0
    num_batches = 0
    
    for x, t in train_loader:
        optimizer.zero_grad()

        x = x.to(device)  # [batch_size, seq_length]
        t = t.to(device)  # [batch_size, seq_length]
    
        y = model(x)
  
        loss = criterion(y.reshape(-1, config.vocab_size), t.reshape(-1))
        acc = accuracy(y, t)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += acc
        num_batches += 1

    if (epoch+1)%10 == 0:
        print(f"{epoch+1}:\tloss:{total_loss/num_batches:.3f}\tacc:{total_acc/num_batches:.3f}")

10:	loss:7.667	acc:0.059


In [ ]:
torch.save({
        "model_state_dict": model.state_dict(),
        "config": config.__dict__,  # configも一緒に保存
        }, pretrain_filename)

## 学習結果の確認・マスク \<mask\> 部分の予想
* 学習したモデルを読み込んでから実行
* 入力できるテキストは、\<mask\>が1箇所のみ
* 複数の\<mask\>をつけるとエラーになります

In [ ]:
import torch.nn.functional as F

# (1) テキストをidsへ
def encode_text_with_special_tokens(text, tokenizer, config):
    """
    テキストをtokenizerに従い、idsへ変換する関数
    text: 例 織田信<mask>は桶狭間の戦いで... : <mask>は1個に限定しておく
    tokenizer: 学習時に使った tokenizer
    戻り値: (1, seq_len) = (1, 64) の torch.LongTensor input_ids
    """
    ids = tokenizer.encode(text).ids                           # 文字列をID化
    ids = [config.bos_token_id] + ids + [config.eos_token_id]  # <bos>, <eos> を追加
    ids = ids[:config.seq_len]                                 # 長すぎる場合は切る
    if len(ids) < config.seq_len:                              # 足りない場合は<pad>
        ids = ids + [config.pad_token_id] * (config.seq_len - len(ids))

    # LongTensorに変換して出力
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    return input_ids

# (2) top 5の表示
@torch.inference_mode()
def predict_mask_topk(model, tokenizer, config, text, topk=5):
    """
    text中の <mask> 位置を見つけて，上位候補を返す
    """
    model.eval()
    input_ids = encode_text_with_special_tokens(text, tokenizer, config)  # textをエンコード化<bos>や<eos>もついているぞ
    logits = model(input_ids).to(device)           # モデル出力: (1, seq_len, vocab_size) = (1, 64, 2000)
    mask_positions = (input_ids[0] == config.mask_token_id).nonzero(as_tuple=True)[0]  # <mask> の位置を探す
    pos = mask_positions.item()                    # mask_positionはtorch.Tensorなので数値に戻す
    mask_logits = logits[0, pos]                   # maski_position位置の語彙方向のスコア logitsを取得 (vocab_size)
    probs = F.softmax(mask_logits, dim=-1)         # 確率化 (自分が解釈するため)
    top_probs, top_ids = torch.topk(probs, k=topk) # 上位 topk 個

    # topkの(ID，トークン，確率)を戻り値に指定
    candidates = []
    for prob, token_id in zip(top_probs.tolist(), top_ids.tolist()):
        token_str = tokenizer.id_to_token(token_id)
        candidates.append((token_id, token_str, prob))
    return {"position": pos,"candidates": candidates}

In [10]:
# 例文
text = "安土桃<mask>時代について"
text = "江戸時代は、徳<mask>家康によって"
#text = "安徳天皇を擁護していてもその即位はクーデターによるものであり、<mask>が自己の立場を正当化することは困難だった。"
#text = "インフレとは、貨<mask>価値が下落することである。"

results = predict_mask_topk(model, tokenizer, config, text, topk=5)

import pandas as pd
print(f"mask position = {results['position']}")
df = pd.DataFrame(results["candidates"] , columns=["id", "token", "prob"])
df

mask position = 6


,id,token,prob
0,954,川,0.992377
1,199,田,0.001144
2,596,良,0.000729
3,45,政,0.000479
4,79,下,0.000374
